In [ ]:
import pandas as pd

raw_data = pd.read_csv('raw_data.csv')
cleaned_data = pd.read_csv('clean_data.csv')

In [ ]:
cleaned_data.columns

In [ ]:
# Define column groups
numerical_cols = [
    'Age at Diagnosis', 'Age at Sx', 'BMI',
    'Tumor Size 1 (cm)', 'Tumor Size 2 (cm)', 'Tumor Size 3 (cm)',
    'Duration of symptoms preop (months)', 'EBL',
    'Mitotic figures in 10 hpf',
    'PFS (After Sx; Months)', 'Overall Survival (yrs)', 'Length of followup (months)', 'LOS (days)'
]

binary_cols = [
    'Sex', 'Cyst?', 'Recurrent Tumor (Y/N)','Ki-67_index_7%', 'Prior surgery', 'Prior Chemo', 'Prior radiation',
    'TERT_marker', 'Synaptophysin_marker', 'GFAP_marker', 'Olig2_marker',
    'Preop Hydrocephalus', 'Preop CN Palsy (Y/N)', 'Preop Papilledema (Y/N)', 'Preop HA',
    'Preop Nausea/Vomiting', 'Preop vertigo', 'Preop Cerebellar Signs', 'Preop Weakness',
    'Preop Sensory Loss', 'Temporary EVD required', 'Shunt placed preop',
    'Shunt placed postop', 'Postop Hydrocephalus', 'New/worse Postop CN Palsy Y/N',
    'Postop Weakness', 'Postop Sensory Changes', 'Postop cerebellar deficit',
    'Postop complication', 'Adjuvant therapy (Y/N)', 'Adjuvant chemo',
    'Radiotherapy (Y/N)', 'Recurrence', 'Death (Y/N)', 'Followup CN Palsy Y/N',
    'Followup Weakness', 'Followup Sensory Changes',
    'Cerebellar function at followup compared to preop', 'Vertigo at followup', 'Intraop Complications'
]

multiclass_cols = [
    'Cranial Location (Infratentorial, supratentorial)',
    'Intra vs Extra Ventricular vs Both', 'Laterality',
    'WHO Grade', 'Enhancement', 'Surgical Approach', 'EOR (GTR, NTR, STR, biopsy)',
    'Discharge dispo'
]

sensitive_col = ['MRN', 'Date of Sx', 'Initials', 'Date of Birth']


In [ ]:
def summarize_numerical(data, cols):
    summary = []
    for col in cols:
        median = data[col].median()
        q1 = data[col].quantile(0.25)
        q3 = data[col].quantile(0.75)
        iqr = q3 - q1
        summary.append({
            'Variable': col,
            'Q1 (25%)': round(q1, 2),
            'Median': round(median, 2),
            'Q3 (75%)': round(q3, 2),
            'IQR': round(iqr, 2)
        })
    return pd.DataFrame(summary).set_index('Variable')


clean_summary = summarize_numerical(cleaned_data, numerical_cols)


In [ ]:
raw_summary = summarize_numerical(raw_data, numerical_cols)

# Combine into one summary table
comparison = clean_summary.join(raw_summary, lsuffix='_clean', rsuffix='_raw')
comparison['Median Δ'] = (comparison['Median_clean'] - comparison['Median_raw']).abs()
comparison['IQR Δ'] = (comparison['IQR_clean'] - comparison['IQR_raw']).abs()
comparison = comparison[['Median_clean', 'Median_raw', 'Median Δ', 'IQR_clean', 'IQR_raw', 'IQR Δ']]


In [ ]:
clean_summary

In [ ]:
comparison

In [ ]:
raw_summary

In [ ]:
from scipy.stats import wilcoxon

statistical_test_results = []

for col in numerical_cols:
    raw_vals = raw_data[col]
    clean_vals = cleaned_data[col]

    # Drop rows with NaNs in either dataset
    paired = pd.DataFrame({
        'raw': raw_vals,
        'clean': clean_vals
    }).dropna()

    # Check if we have at least 5 valid pairs
    if len(paired) >= 5:
        try:
            # Check for actual differences (Wilcoxon skips if all diffs are 0)
            if (paired['raw'] != paired['clean']).any():
                stat, p = wilcoxon(paired['raw'], paired['clean'])
                statistical_test_results.append({
                    'Variable': col,
                    'p-value': p,
                    'Significant': p < 0.05
                })
            else:
                statistical_test_results.append({
                    'Variable': col,
                    'p-value': 1.0,
                    'Significant': False
                })
        except Exception as e:
            statistical_test_results.append({
                'Variable': col,
                'p-value': None,
                'Significant': None
            })
    else:
        statistical_test_results.append({
            'Variable': col,
            'p-value': None,
            'Significant': None
        })

stats_df = pd.DataFrame(statistical_test_results).set_index('Variable')


In [ ]:
stats_df

In [ ]:
def summarize_binary(data, cols, positive_values=['Y', 'Yes', 1, '1', 'True', 'T']):
    summary = []

    for col in cols:
        counts = data[col].value_counts(dropna=False)
        total = counts.sum()

        # Log categories for debugging (optional)
        # print(f"{col}: {list(counts.index)}")

        # Try to find a matching positive value
        positive = 0
        for val in positive_values:
            if val in counts:
                positive = counts[val]
                break

        percent = (positive / total) * 100 if total > 0 else None

        summary.append({
            'Variable': col,
            'Count (Y)': positive,
            'Total': total,
            'Percent (Y)': round(percent, 1) if percent is not None else None
        })

    return pd.DataFrame(summary).set_index('Variable')

binary_summary = summarize_binary(cleaned_data, binary_cols)
binary_summary = binary_summary.sort_values('Percent (Y)', ascending=False)


In [ ]:
binary_summary = summarize_binary(cleaned_data, binary_cols)
binary_summary = binary_summary.sort_values('Percent (Y)', ascending=False)
display(binary_summary)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# --- Input ---
selected_features = {
    'Cyst?': 'Cystic Component',
    'Ki-67_index_7%': 'Ki-67 > 7%',
    'Synaptophysin_marker': 'Synaptophysin+',
    'TERT_marker': 'TERT+',
    'GFAP_marker': 'GFAP+',
    'Olig2_marker': 'Olig2+'
}

summary_data = []
for col, label in selected_features.items():
    counts = cleaned_data[col].value_counts()
    yes_count = counts.get('Y', counts.get(1, 0))
    no_count = counts.get('N', counts.get(0, 0))
    total = yes_count + no_count
    percent_yes = round((yes_count / total) * 100, 1) if total > 0 else 0
    percent_no = 100 - percent_yes
    summary_data.append({
        'Feature': label,
        '% Yes': percent_yes,
        '% No': percent_no,
        'Yes Count': yes_count,
        'No Count': no_count
    })

df_summary = pd.DataFrame(summary_data).set_index('Feature')

# --- Plotting ---
fig, ax = plt.subplots(figsize=(8, 4), dpi=300)

# Plot only the 'Yes' percentage
bars = ax.barh(df_summary.index, df_summary['% Yes'], color='#4CAF50')

# Add text: Yes count inside the bar
for bar, count in zip(bars, df_summary['Yes Count']):
    ax.text(bar.get_width() - 5, bar.get_y() + bar.get_height() / 2,
            f'{count}', va='center', ha='right', color='white', fontsize=10)

# Add % Yes as label on right end
for bar, pct in zip(bars, df_summary['% Yes']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2,
            f'{pct}%', va='center', ha='left', color='black', fontsize=10)

ax.set_xlim(0, 100)
ax.set_xlabel('Percentage of Patients (%)')
ax.set_title('Feature and Marker Positivity Rates')
ax.grid(axis='x', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = raw_data.copy()

# Define symptom progression across stages (some may not have follow-up)
symptoms = {
    'Hydrocephalus': [
        'Preop Hydrocephalus',
        'Postop Hydrocephalus',
        None
    ],
    'Cranial Nerve Palsy': [
        'Preop CN Palsy (Y/N)',
        'New/worse Postop CN Palsy Y/N',
        'Followup CN Palsy Y/N'
    ],
    'Cerebellar Signs': [
        'Preop Cerebellar Signs',
        'Postop cerebellar deficit',
        'Cerebellar function at followup compared to preop'
    ],
    'Nausea / Vomiting': [
        'Preop Nausea/Vomiting',
        None,
        None
    ],
    'Weakness': [
        'Preop Weakness',
        'Postop Weakness',
        'Followup Weakness'
    ],
    'Sensory Loss': [
        'Preop Sensory Loss',
        'Postop Sensory Changes',
        'Followup Sensory Changes'
    ],
    'Vertigo': [
        'Preop vertigo',
        None,
        'Vertigo at followup'
    ],
    'Papilledema': [
        'Preop Papilledema (Y/N)',
        None,
        None
    ],
    'Intraop Complications': [
        None,
        'Intraop Complications',
        None
    ]
}

stages = ['Pre-op', 'Post-op', 'Follow-up']
plt.figure(figsize=(10, 6), dpi=300)

for symptom, cols in symptoms.items():
    values = []
    for col in cols:
        if col and col in df.columns:
            values.append(df[col].eq('Y').sum())
        else:
            values.append(None)
    plt.plot(stages, values, marker='o', linewidth=2, label=symptom)

plt.ylim(0, 18)
plt.ylabel('Number of Patients')
plt.title('Symptom Progression Across Clinical Course')
plt.grid(True, linestyle='--', alpha=0.4)
plt.legend(title='Symptom', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import string

df = raw_data.copy()

# Define stages and symptom progression columns
stages = ['Pre-op', 'Post-op', 'Follow-up']

symptoms = {
    'Hydrocephalus': ['Preop Hydrocephalus', 'Postop Hydrocephalus', None],
    'Cranial Nerve Palsy': ['Preop CN Palsy (Y/N)', 'New/worse Postop CN Palsy Y/N', 'Followup CN Palsy Y/N'],
    'Cerebellar Signs': ['Preop Cerebellar Signs', 'Postop cerebellar deficit', 'Cerebellar function at followup compared to preop'],
    'Nausea / Vomiting': ['Preop Nausea/Vomiting', None, None],
    'Weakness': ['Preop Weakness', 'Postop Weakness', 'Followup Weakness'],
    'Sensory Loss': ['Preop Sensory Loss', 'Postop Sensory Changes', 'Followup Sensory Changes'],
    'Vertigo': ['Preop vertigo', None, 'Vertigo at followup'],
    'Papilledema': ['Preop Papilledema (Y/N)', None, None],
    'Intraop Complications': [None, 'Intraop Complications', None]
}

# Layout
n_symptoms = len(symptoms)
n_cols = 3
n_rows = -(-n_symptoms // n_cols)

fig, axs = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows), dpi=300, sharey=True)
axs = axs.flatten()

panel_labels = list(string.ascii_uppercase)  # ['A', 'B', 'C', ...]

for i, (symptom, cols) in enumerate(symptoms.items()):
    values = []
    valid_stages = []

    for stage, col in zip(stages, cols):
        if col and col in df.columns:
            count = df[col].eq('Y').sum()
            values.append(count)
            valid_stages.append(stage)

    ax = axs[i]

    # Plotting logic
    if len(values) == 1:
        ax.bar(valid_stages, values, color="#4C72B0", alpha=0.8)
    elif len(values) >= 2:
        ax.plot(valid_stages, values, marker='o', linewidth=2, color="#4C72B0")
        ax.fill_between(valid_stages, values, alpha=0.2, color="#4C72B0")

    ax.set_title(symptom)
    ax.set_ylim(0, 18)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xticks(stages)

    # Add panel label (A, B, C...) in lower-left corner
    ax.text(0.01, 0.02, panel_labels[i], transform=ax.transAxes,
            fontsize=12, fontweight='bold', va='bottom', ha='left')

# Hide any unused axes
for j in range(i + 1, len(axs)):
    axs[j].axis('off')

fig.suptitle('Symptom Trajectories Across Clinical Course', fontsize=16)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()


In [ ]:
# --- 0) Build `synthetic_datasets` from CSVs in synthetic_data/ ---
import os
import pandas as pd
import numpy as np

os.makedirs("figures", exist_ok=True)

SYN_DIR = "synthetic_data"
MODEL_FILES = {
    "GaussianCopula": "GaussianCopula_withanon.csv",
    "CopulaGAN": "CopulaGAN_withanon.csv",
    "CTGAN": "CTGAN_withanon.csv",
    "TVAE": "TVAE_withanon.csv"
}

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import string

# --- Load synthetic datasets ---
synthetic_datasets = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(SYN_DIR, fname)
    synthetic_datasets[name] = pd.read_csv(path)

# Use same stages & symptom mapping
stages = ['Pre-op', 'Post-op', 'Follow-up']

symptoms = {
    'Hydrocephalus': ['Preop Hydrocephalus', 'Postop Hydrocephalus', None],
    'Cranial Nerve Palsy': ['Preop CN Palsy (Y/N)', 'New/worse Postop CN Palsy Y/N', 'Followup CN Palsy Y/N'],
    'Cerebellar Signs': ['Preop Cerebellar Signs', 'Postop cerebellar deficit', 'Cerebellar function at followup compared to preop'],
    'Nausea / Vomiting': ['Preop Nausea/Vomiting', None, None],
    'Weakness': ['Preop Weakness', 'Postop Weakness', 'Followup Weakness'],
    'Sensory Loss': ['Preop Sensory Loss', 'Postop Sensory Changes', 'Followup Sensory Changes'],
    'Vertigo': ['Preop vertigo', None, 'Vertigo at followup'],
    'Papilledema': ['Preop Papilledema (Y/N)', None, None],
    'Intraop Complications': [None, 'Intraop Complications', None]
}

panel_labels = list(string.ascii_uppercase)

# --- Plot per synthetic model ---
for model_name, df_syn in synthetic_datasets.items():

    n_symptoms = len(symptoms)
    n_cols = 3
    n_rows = -(-n_symptoms // n_cols)

    fig, axs = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows),
                            dpi=300, sharey=True)
    axs = axs.flatten()

    for i, (symptom, cols) in enumerate(symptoms.items()):
        values = []
        valid_stages = []

        for stage, col in zip(stages, cols):
            if col and col in df_syn.columns:
                count = df_syn[col].eq('Y').sum()
                values.append(count)
                valid_stages.append(stage)

        ax = axs[i]

        if len(values) == 1:
            ax.bar(valid_stages, values, alpha=0.8)
        elif len(values) >= 2:
            ax.plot(valid_stages, values, marker='o', linewidth=2)
            ax.fill_between(valid_stages, values, alpha=0.2)

        ax.set_title(symptom)
        ax.set_ylim(0, df_syn.shape[0])  # scale to synthetic cohort size
        ax.grid(True, linestyle='--', alpha=0.3)
        ax.set_xticks(stages)

        ax.text(0.01, 0.02, panel_labels[i], transform=ax.transAxes,
                fontsize=12, fontweight='bold', va='bottom', ha='left')

    for j in range(i + 1, len(axs)):
        axs[j].axis('off')

    fig.suptitle(f'Synthetic Symptom Trajectories – {model_name}', fontsize=16)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f'figures/symptom_trajectory_{model_name}.png', dpi=300)
    plt.show()


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import string
import os

# --- Color palette per synthesizer ---
model_colors = {
    'GaussianCopula': '#1f77b4',
    'CopulaGAN': '#ff7f0e',
    'CTGAN': '#2ca02c',
    'TVAE': '#d62728'
}

# --- Load synthetic datasets ---
synthetic_datasets = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(SYN_DIR, fname)
    synthetic_datasets[name] = pd.read_csv(path)

# Stages and symptom mapping (unchanged)
stages = ['Pre-op', 'Post-op', 'Follow-up']

symptoms = {
    'Hydrocephalus': ['Preop Hydrocephalus', 'Postop Hydrocephalus', None],
    'Cranial Nerve Palsy': ['Preop CN Palsy (Y/N)', 'New/worse Postop CN Palsy Y/N', 'Followup CN Palsy Y/N'],
    'Cerebellar Signs': ['Preop Cerebellar Signs', 'Postop cerebellar deficit', 'Cerebellar function at followup compared to preop'],
    'Nausea / Vomiting': ['Preop Nausea/Vomiting', None, None],
    'Weakness': ['Preop Weakness', 'Postop Weakness', 'Followup Weakness'],
    'Sensory Loss': ['Preop Sensory Loss', 'Postop Sensory Changes', 'Followup Sensory Changes'],
    'Vertigo': ['Preop vertigo', None, 'Vertigo at followup'],
    'Papilledema': ['Preop Papilledema (Y/N)', None, None],
    'Intraop Complications': [None, 'Intraop Complications', None]
}

panel_labels = list(string.ascii_uppercase)

# --- Layout ---
n_symptoms = len(symptoms)
n_cols = 3
n_rows = -(-n_symptoms // n_cols)

fig, axs = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows),
                        dpi=300, sharey=True)
axs = axs.flatten()

# --- Plot ---
for i, (symptom, cols) in enumerate(symptoms.items()):
    ax = axs[i]

    for model_name, df_syn in synthetic_datasets.items():
        values = []
        valid_stages = []

        for stage, col in zip(stages, cols):
            if col and col in df_syn.columns:
                # normalize to percentage to compare shapes
                pct = df_syn[col].eq('Y').mean() * 100
                values.append(pct)
                valid_stages.append(stage)

        if len(values) >= 2:
            ax.plot(
                valid_stages,
                values,
                marker='o',
                linewidth=2,
                color=model_colors[model_name],
                alpha=0.9,
                label=model_name if i == 0 else None  # legend once
            )
            ax.fill_between(
                valid_stages,
                values,
                color=model_colors[model_name],
                alpha=0.15
            )
        elif len(values) == 1:
            ax.scatter(
                valid_stages,
                values,
                s=60,
                color=model_colors[model_name],
                alpha=0.9,
                label=model_name if i == 0 else None
            )

    ax.set_title(symptom)
    ax.set_ylim(0, 100)
    ax.grid(True, linestyle='--', alpha=0.3)
    ax.set_xticks(stages)
    ax.set_ylabel('% of Patients')

    # Panel label
    ax.text(0.01, 0.02, panel_labels[i],
            transform=ax.transAxes,
            fontsize=12, fontweight='bold',
            va='bottom', ha='left')

# Hide unused axes
for j in range(i + 1, len(axs)):
    axs[j].axis('off')

# --- Global legend ---
handles, labels = axs[0].get_legend_handles_labels()
fig.legend(
    handles, labels,
    loc='upper center',
    bbox_to_anchor=(0.5, 0.965),  # ⬅ move legend down
    ncol=len(model_colors),
    frameon=False
)

fig.suptitle('Synthetic Symptom Trajectories Across Clinical Course',
             fontsize=16)

plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.savefig('figures/synthetic_symptom_trajectories_all_models.png',
            dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import fisher_exact, chi2_contingency

# Binary variables to report
BINARY_VARS = {
    'Hydrocephalus (Pre-op)': 'Preop Hydrocephalus',
    'CN Palsy (Pre-op)': 'Preop CN Palsy (Y/N)',
    'Cerebellar Signs (Pre-op)': 'Preop Cerebellar Signs',
    'Weakness (Pre-op)': 'Preop Weakness',
    'Sensory Loss (Pre-op)': 'Preop Sensory Loss',
    'Vertigo (Pre-op)': 'Preop vertigo',
    'Papilledema (Pre-op)': 'Preop Papilledema (Y/N)',
    'Cystic Component': 'Cyst?',
    'Olig2+': 'Olig2_marker',
    'GFAP+': 'GFAP_marker',
    'Synaptophysin+': 'Synaptophysin_marker',
    'TERT+': 'TERT_marker',
    'Ki-67 ≥7%': 'Ki-67_index_7%'
}

def count_percent(df, col):
    """Return (count, percent) for Y/1 values."""
    if col not in df.columns:
        return np.nan, np.nan
    series = df[col].astype(str)
    count = series.eq('Y').sum()
    total = series.notna().sum()
    percent = 100 * count / total if total > 0 else np.nan
    return count, percent


def p_value_vs_raw(raw_df, syn_df, col):
    """Fisher exact if sparse, otherwise chi-square."""
    if col not in raw_df.columns or col not in syn_df.columns:
        return np.nan

    raw_yes = raw_df[col].astype(str).eq('Y').sum()
    raw_no  = raw_df[col].astype(str).ne('Y').sum()

    syn_yes = syn_df[col].astype(str).eq('Y').sum()
    syn_no  = syn_df[col].astype(str).ne('Y').sum()

    table = [[raw_yes, raw_no], [syn_yes, syn_no]]

    # Fisher for small counts
    if min(raw_yes, raw_no, syn_yes, syn_no) < 5:
        _, p = fisher_exact(table)
    else:
        _, p, _, _ = chi2_contingency(table)

    return p

# Load raw data
raw_df = pd.read_csv('raw_data.csv')

# Load synthetic datasets
synthetic_datasets = {}
for name, fname in MODEL_FILES.items():
    synthetic_datasets[name] = pd.read_csv(os.path.join(SYN_DIR, fname))

rows = []

for label, col in BINARY_VARS.items():

    raw_n, raw_pct = count_percent(raw_df, col)
    row = {
        'Variable': label,
        'Raw N (%)': f"{raw_n} ({raw_pct:.1f}%)"
    }

    for model_name, syn_df in synthetic_datasets.items():
        syn_n, syn_pct = count_percent(syn_df, col)
        p = p_value_vs_raw(raw_df, syn_df, col)

        row[f'{model_name} N (%)'] = f"{syn_n} ({syn_pct:.1f}%)"
        row[f'{model_name} p-value'] = p

    rows.append(row)

summary_table = pd.DataFrame(rows)
summary_table
